In [1]:
pip install dash plotly pandas openpyxl

   ---------------------------------------- 0.0/7.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/7.2 MB ? eta -:--:--
   ----- ---------------------------------- 1.0/7.2 MB 3.7 MB/s eta 0:00:02
   ---------- ----------------------------- 1.8/7.2 MB 4.1 MB/s eta 0:00:02
   ----------------- ---------------------- 3.1/7.2 MB 4.4 MB/s eta 0:00:01
   ------------------ --------------------- 3.4/7.2 MB 4.3 MB/s eta 0:00:01
   ----------------------- ---------------- 4.2/7.2 MB 3.9 MB/s eta 0:00:01
   -------------------------- ------------- 4.7/7.2 MB 3.5 MB/s eta 0:00:01
   ------------------------------ --------- 5.5/7.2 MB 3.6 MB/s eta 0:00:01
   ---------------------------------- ----- 6.3/7.2 MB 3.5 MB/s eta 0:00:01
   ---------------------------------------- 7.2/7.2 MB 3.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: C:\Users\rajmu\anaconda3\anaconda\python.exe -m pip install --upgrade pip


In [4]:

import pandas as pd
import numpy as np

import dash
from dash import dcc, html
from dash.dependencies import Input, Output

import plotly.express as px

# LOAD DATA

In [5]:
monthly_trend = pd.read_csv(
    "monthly_prescribing_trend.csv"
)

In [6]:
icb_summary = pd.read_csv(
    "icb_prescribing_summary.csv"
)

In [7]:
drug_summary = pd.read_csv(
    "drug_prescribing_summary.csv"
)

In [8]:
dropoff_df = pd.read_csv(
    "dropoff_rate_by_region.csv"
)

# DATE CLEANING

In [10]:
monthly_trend["Month"] = pd.to_datetime(
    monthly_trend["Month"]
)



# DASH APP

In [11]:
app = dash.Dash(__name__)

app.title = "NHS Mental Health Dashboard"

# DROPDOWN OPTIONS

In [12]:

region_options = [
    {"label": region, "value": region}
    for region in sorted(
        icb_summary["ICB_NAME"].dropna().unique()
    )
]

drug_options = [
    {"label": drug, "value": drug}
    for drug in sorted(
        monthly_trend["Drug_Category"].dropna().unique()
    )
]


# APP LAYOUT

In [20]:
app.layout = html.Div([

    html.H1(
        "NHS Mental Health Analytics Dashboard",
        style={
            "textAlign": "center"
        }
    ),

    # ========================================================
    # FILTERS
    # ========================================================

    html.Div([

        html.Div([

            html.Label("Select Drug Category"),

            dcc.Dropdown(
                id="drug_dropdown",
                options=drug_options,
                value="Antidepressant",
                clearable=False
            )

        ], style={
            "width": "30%",
            "display": "inline-block",
            "padding": "10px"
        }),

        html.Div([

            html.Label("Select Region"),

            dcc.Dropdown(
                id="region_dropdown",
                options=region_options,
                value=icb_summary["ICB_NAME"].iloc[0],
                clearable=False
            )

        ], style={
            "width": "40%",
            "display": "inline-block",
            "padding": "10px"
        })

    ]),

    # ========================================================
    # KPI CARDS
    # ========================================================

    html.Div([

        html.Div([
            html.H3("Total Prescriptions"),
            html.H2(id="kpi_total_items")
        ], className="card"),

        html.Div([
            html.H3("Total Cost"),
            html.H2(id="kpi_total_cost")
        ], className="card"),

        html.Div([
            html.H3("Drop-off Rate"),
            html.H2(id="kpi_dropoff")
        ], className="card"),

    ], style={
        "display": "flex",
        "justifyContent": "space-around",
        "margin": "20px"
    }),

    # ========================================================
    # CHARTS
    # ========================================================

    dcc.Graph(id="monthly_trend_chart"),

    dcc.Graph(id="cost_trend_chart"),

    dcc.Graph(id="top_medicines_chart"),

    dcc.Graph(id="forecast_chart"),

    dcc.Graph(id="funnel_chart"),

])




# ============================================================
# FORECAST, FUNNEL AND DROPOFF DATA
# ============================================================

forecast_df = monthly_trend.copy()

forecast_df["Forecast_Total_Items"] = (
    forecast_df["Total_Items"] * 1.05
)

funnel_df = pd.DataFrame({
    "Stage": [
        "Prescribed",
        "Dispensed",
        "Completed Treatment"
    ],
    "Count": [
        monthly_trend["Total_Items"].sum(),
        monthly_trend["Total_Items"].sum() * 0.85,
        monthly_trend["Total_Items"].sum() * 0.65
    ]
})

dropoff_df = pd.DataFrame({
    "Dropoff_Rate": [35.0]
})







# ============================================================
# CALLBACKS
# ============================================================

@app.callback(

    [
        Output("monthly_trend_chart", "figure"),
        Output("cost_trend_chart", "figure"),
        Output("top_medicines_chart", "figure"),
        Output("forecast_chart", "figure"),
        Output("funnel_chart", "figure"),
        Output("kpi_total_items", "children"),
        Output("kpi_total_cost", "children"),
        Output("kpi_dropoff", "children"),
    ],

    [
        Input("drug_dropdown", "value"),
        Input("region_dropdown", "value")
    ]

)

def update_dashboard(selected_drug, selected_region):

    # ========================================================
    # FILTER MONTHLY TREND
    # ========================================================

    filtered_monthly = monthly_trend[
        monthly_trend["Drug_Category"] == selected_drug
    ]

    # ========================================================
    # MONTHLY TREND FIGURE
    # ========================================================

    fig_monthly = px.line(
        filtered_monthly,
        x="Month",
        y="Total_Items",
        title=f"{selected_drug} Monthly Prescribing Trend",
        markers=True
    )

    # ========================================================
    # COST TREND
    # ========================================================

    fig_cost = px.line(
        filtered_monthly,
        x="Month",
        y="Total_Cost",
        title=f"{selected_drug} Cost Trend",
        markers=True
    )

    # ========================================================
    # TOP MEDICINES
    # ========================================================

    filtered_drugs = drug_summary[
        drug_summary["Drug_Category"] == selected_drug
    ].head(15)

    fig_drugs = px.bar(
        filtered_drugs,
        x="BNF_CHEMICAL_SUBSTANCE",
        y="Total_Items",
        title=f"Top Medicines - {selected_drug}"
    )

    # ========================================================
    # FORECAST
    # ========================================================

    filtered_forecast = forecast_df[
        forecast_df["Drug_Category"] == selected_drug
    ]

    fig_forecast = px.line(
        filtered_forecast,
        x="Month",
        y="Forecast_Total_Items",
        title=f"{selected_drug} Forecast",
        markers=True
    )

    # ========================================================
    # FUNNEL
    # ========================================================

    fig_funnel = px.funnel(
        funnel_df,
        x="Count",
        y="Stage",
        title="Treatment Funnel"
    )

    # ========================================================
    # KPI VALUES
    # ========================================================

    total_items = (
        filtered_monthly["Total_Items"].sum()
    )

    total_cost = (
        filtered_monthly["Total_Cost"].sum()
    )

    dropoff = (
        dropoff_df["Dropoff_Rate"].mean()
    )

    return (
        fig_monthly,
        fig_cost,
        fig_drugs,
        fig_forecast,
        fig_funnel,
        f"{total_items:,.0f}",
        f"£{total_cost:,.2f}",
        f"{dropoff:.2f}%"
    )

# ============================================================
# RUN APP
# ============================================================

if __name__ == "__main__":
    app.run(
        debug=False,
        port=8050,
        jupyter_mode="external"
    )

Dash app running on http://127.0.0.1:8050/
---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
Cell In[16], line 194, in update_dashboard(
    selected_drug='Antidepressant',
    selected_region='NHS NORTH EAST AND NORTH CUMBRIA INTEGRATED CARE BOARD'
)
    182 total_cost = (
    183     filtered_monthly["Total_Cost"].sum()
    184 )
    186 dropoff = (
    187     dropoff_df["Dropoff_Rate"].mean()
    188 )
    190 return (
    191     fig_monthly,
    192     fig_cost,
    193     fig_drugs,
--> 194     fig_forecast,
        fig_monthly = Figure({
    'data': [{'hovertemplate': 'Month=%{x}<br>Total_Items=%{y}<extra></extra>',
              'legendgroup': '',
              'line': {'color': '#636efa', 'dash': 'solid'},
              'marker': {'symbol': 'circle'},
              'mode': 'lines+markers',
              'name': '',
              'orientation': 'v',
              'showlegend